# Product Data Quality Workflow

**Author: Feras Elghandour**

Turn incomplete product metadata into a consistent, filterable product shortlist and auditable exports.

A deterministic synthetic product/review fixture, missing-price handling, rating filters, guarded lookups, CSV/Parquet export, and exploratory pricing charts.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# Set random seed for reproducibility
np.random.seed(42)

print('All libraries imported successfully!')

In [ ]:
# Note: Using simulated data that mirrors Hugging Face Amazon Reviews 2023 structure
# In production: from datasets import load_dataset

sample_limit = 100  # Integer value of 100

product_titles = [
    "Sony WH-1000XM5 Wireless Headphones", "Apple AirPods Pro 2nd Gen",
    "Samsung 65-inch QLED TV", "Bose QuietComfort 45", "Anker PowerCore 26800",
    "Logitech MX Master 3 Mouse", "iPad Pro 12.9-inch M2", "Dell XPS 15 Laptop",
    "Sony PlayStation 5", "Nintendo Switch OLED", "Amazon Echo Dot 5th Gen",
    "Google Pixel 7 Pro", "Samsung Galaxy S23 Ultra", "Apple Watch Series 8",
    "Kindle Paperwhite 11th Gen", "Ring Video Doorbell Pro 2", "Nest Learning Thermostat",
    "GoPro HERO11 Black", "DJI Mini 3 Pro Drone", "Razer BlackWidow Keyboard",
    "LG OLED C2 55-inch TV", "Apple Mac Mini M2", "Corsair K95 RGB Keyboard",
    "SteelSeries Arctis Nova Pro", "Jabra Evolve2 85 Headset", "HP OfficeJet Pro 9015",
    "Canon EOS R6 Mark II", "Nikon Z6 II Camera", "Sony A7 IV Mirrorless",
    "Dyson V15 Detect Vacuum", "iRobot Roomba j7+", "Instant Pot Duo 7-in-1",
    "Vitamix 5200 Blender", "KitchenAid Stand Mixer", "Ninja Foodi Air Fryer",
    "Philips Hue Starter Kit", "Lutron Caseta Dimmer", "TP-Link Kasa Smart Plug",
    "Eero Pro 6E Mesh WiFi", "Netgear Orbi RBK863S", "Western Digital 4TB HDD",
    "Samsung 970 EVO Plus SSD", "Crucial 32GB DDR5 RAM", "EVGA GeForce RTX 3080",
    "AMD Ryzen 9 5900X CPU", "ASUS ROG Strix Z690-E", "Corsair RM850x PSU",
    "Fractal Design Meshify C", "be quiet! Dark Rock Pro 4", "Noctua NH-D15 Cooler",
    "LG 27GP850-B Monitor", "Dell S2722QC Monitor", "BenQ EW3280U Monitor",
    "Anker USB-C Hub 10-in-1", "CalDigit TS3 Plus Dock", "Elgato Stream Deck",
    "Blue Yeti USB Microphone", "Audio-Technica AT2020", "Focusrite Scarlett 2i2",
    "Behringer U-PHORIA UM2", "Shure SM7B Microphone", "RODE NT1 5th Gen",
    "Sony ZV-E10 Vlog Camera", "Fujifilm X-T5 Camera", "Panasonic GH6 Camera",
    "DJI Osmo Mobile 6", "Zhiyun Crane M3 Gimbal", "Insta360 One X3",
    "Peak Design Everyday Backpack", "Pelican 1510 Carry-On", "Lowepro ProTactic 450",
    "Moment Wide Lens 18mm", "Moment Tele Lens 58mm", "Sandisk Extreme Pro 256GB",
    "Lexar 1066x 128GB CF", "Manfrotto 055 Tripod", "Joby GorillaPod 5K",
    "Profoto B10 Plus Flash", "Godox AD200Pro Flash", "Neewer 660 LED Panel",
    "Elgato Key Light Air", "Logitech C920 Webcam", "Razer Kiyo Pro Webcam",
    "NVIDIA Shield TV Pro", "Apple TV 4K 3rd Gen", "Roku Ultra 2022",
    "Amazon Fire TV Stick 4K", "Chromecast with Google TV", "Xbox Series X",
    "PlayStation VR2", "Meta Quest Pro", "HTC Vive Pro 2",
    "Valve Index VR Kit", "Oculus Quest 2 128GB", "Samsung Galaxy Tab S8",
    "Microsoft Surface Pro 9", "Lenovo ThinkPad X1 Carbon", "ASUS ZenBook Pro 14",
    "HP Spectre x360 14", "Razer Blade 15 Gaming", "MSI Creator Z16",
    "Framework Laptop 13"
]

reviews_data = []
i = 0
for i in range(sample_limit + 1):
    if i >= sample_limit:
        break
    reviews_data.append({
        'asin': f'B{str(i+1000).zfill(9)}',
        'user_id': f'USER_{str(i+1).zfill(5)}',
        'rating': float(np.random.choice([3.0,3.5,4.0,4.5,5.0], p=[0.05,0.10,0.25,0.35,0.25])),
        'title': f"Great product - {'highly recommend' if i%3==0 else 'works as expected' if i%3==1 else 'excellent quality'}",
        'text': f"This is review number {i+1}. The product exceeded my expectations in many ways.",
        'verified_purchase': bool(np.random.choice([True, False], p=[0.85, 0.15])),
        'helpful_votes': int(np.random.randint(0, 50)),
        'timestamp': f"2023-{str(np.random.randint(1,13)).zfill(2)}-{str(np.random.randint(1,28)).zfill(2)}"
    })

reviews_df = pd.DataFrame(reviews_data)
print(f'Generating synthetic review fixture ({sample_limit} rows)...')
print(f'Reviews dataset loaded successfully: {reviews_df.shape[0]} rows × {reviews_df.shape[1]} columns')

In [ ]:
print('First 10 items of the Reviews DataFrame:')

Items_To_Print = 10
pd.set_option('display.max_rows', Items_To_Print)
display(reviews_df.head(Items_To_Print))

In [ ]:
metadata_limit = 100  # Integer value of 100

metadata_data = []
j = 0
# For loop up to metadata_limit then break
for j in range(metadata_limit + 1):
    if j >= metadata_limit:
        break
    has_price = np.random.random() > 0.12
    price = round(float(np.random.uniform(9.99, 999.99)), 2) if has_price else None
    metadata_data.append({
        'asin': f'B{str(j+1000).zfill(9)}',
        'title': product_titles[j] if j < len(product_titles) else f'Electronic Product {j+1}',
        'price': price,
        'average_rating': round(float(np.random.uniform(3.8, 5.0)), 1),
        'rating_number': int(np.random.randint(50, 50000)),
        'features': f'Feature list for product {j+1}',
        'description': f'High-quality electronic product for modern users. Product {j+1}.',
        'store': str(np.random.choice(['Amazon','Best Buy','Manufacturer Direct','Third Party'])),
        'categories': str(np.random.choice(['Electronics','Computers','Camera & Photo','Audio'])),
        'main_category': 'Electronics'
    })

item_metadata_df = pd.DataFrame(metadata_data)
item_metadata_df['price'] = pd.to_numeric(item_metadata_df['price'], errors='coerce')

print(f'Generating synthetic metadata fixture ({metadata_limit} rows)...')
print(f'Metadata dataset loaded successfully: {item_metadata_df.shape[0]} rows × {item_metadata_df.shape[1]} columns')

In [ ]:
print('Metadata DataFrame head:')
print(item_metadata_df[['asin','title','price','average_rating']].head().to_string())

In [ ]:
def get_product(asin, metadata_df):
    """
    Look up a product by ASIN and return its title and price.
    Uses try-except to handle missing products and null prices.
    """
    try:
        product = metadata_df[metadata_df['asin'] == asin].iloc[0]
        title = product['title']
        price = product['price']
        if pd.isna(price):
            return f"Product '{title}' found but price is not available."
        return f"Product '{title}' found! Price: ${price:.2f}"
    except (IndexError, KeyError) as e:
        return f"Error: Product with ASIN {asin} not found."

print('=== Part 3.3: Exception Handling Demo ===')
print(get_product('B000001000', item_metadata_df))   # Valid product
print(get_product('B999999999', item_metadata_df))   # Invalid ASIN

# Review items with and without price
with_price    = item_metadata_df['price'].notna().sum()
without_price = item_metadata_df['price'].isna().sum()
print(f'\nProducts WITH price: {with_price}')
print(f'Products WITHOUT price: {without_price}')

In [ ]:
print('=== Part 4.1: Dataset Columns ===')

for df_name, df in [('reviews_df', reviews_df), ('item_metadata_df', item_metadata_df)]:
    print(f'\n{df_name} columns:')
    for col in df.columns:
        print(f'  - {col}')

In [ ]:
print('=== Part 4.3: Data Types and Info ===')
print('\nreviews_df dtypes:')
print(reviews_df.dtypes)
print(f'Shape: {reviews_df.shape}')

print('\nitem_metadata_df dtypes:')
print(item_metadata_df.dtypes)
print(f'Shape: {item_metadata_df.shape}')

In [ ]:
print('=== Part 5.1: Cleaning Data ===')

# Set average_rating_filter as float data type
average_rating_filter = 4.5  # float

# Specify the 3 columns for the cleaned DataFrame
item_metadata_cleaned_df = item_metadata_df[['asin', 'title', 'price', 'average_rating']].copy()

# Create top_products_df with 3 parameters:
# 1. has a title
# 2. average_rating >= 4.5
# 3. has a price value
top_products_df = item_metadata_cleaned_df[
    (item_metadata_cleaned_df['title'].notna()) &
    (item_metadata_cleaned_df['title'] != '') &
    (item_metadata_cleaned_df['average_rating'] >= average_rating_filter) &
    (item_metadata_cleaned_df['price'].notna())
].copy()

print(f'Filtering criteria: average_rating >= {average_rating_filter}, has title, has price')
print(f'\nOriginal metadata rows: {len(item_metadata_df)}')
print(f'Top products rows after cleaning: {len(top_products_df)}')
print(f'\nTop 5 products:')
print(top_products_df[['title','price','average_rating']].head().to_string())

In [ ]:
def calculate_percentage(part, whole):
    """
    Calculate the percentage of part relative to whole.
    Returns 0.0 if whole is zero to avoid division by zero error.
    """
    if whole == 0:
        return 0.0
    return (part / whole) * 100

print('=== Part 5.3: Percentage of Cleaned Data ===')
percentage_retained = calculate_percentage(len(top_products_df), len(item_metadata_df))
print(f'Percentage of data retained after cleaning: {percentage_retained:.2f}%')

In [ ]:
# Parts 5.4 & 5.5: Export the cleaned dataset
print('=== Parts 5.4 & 5.5: Exporting Dataset ===')

top_products_df.to_csv('top_products.csv', index=False)
print('Saved top_products.csv')

top_products_df.to_parquet('top_products.parquet', index=False)
print('Saved top_products.parquet')

In [ ]:
print('=== Part 5.6: Top Rated Product Titles ===')

# Create an empty array (list)
top_rated_titles = []

# For loop to iterate over top_products_df and append titles
for idx, row in top_products_df.iterrows():
    top_rated_titles.append(row['title'])

print(f'Total top rated products: {len(top_rated_titles)}')
print('\nTop rated titles:')
for i, title in enumerate(top_rated_titles[:10], 1):
    print(f'  {i:2d}. {title}')

In [ ]:
print('=== Part 6.1: Histogram of Product Prices ===')

plt.figure(figsize=(10, 6))
sns.histplot(data=top_products_df, x='price', kde=True, color='steelblue', bins=15)
plt.title('Distribution of Prices for Highly Rated Products (Rating ≥ 4.5)', fontsize=14)
plt.xlabel('Price ($)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Part 6.2: Scatterplot of Price vs Rating ===')

plt.figure(figsize=(10, 6))
sns.scatterplot(data=top_products_df, x='price', y='average_rating', color='coral', alpha=0.8, s=80)
plt.title('Price vs. Average Rating for Highly Rated Products', fontsize=14)
plt.xlabel('Price ($)', fontsize=12)
plt.ylabel('Average Rating', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Part 6.3: Combined Dashboard ===')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=top_products_df, x='price', kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Price Distribution (Highly Rated Products)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

sns.scatterplot(data=top_products_df, x='price', y='average_rating', ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('Price vs. Average Rating')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Average Rating')

plt.suptitle('Transformational AI — Electronics Product Analysis Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('=== Part 6.4 & 6.5: Highly Rated Products Dashboard ===')

# Cutoff threshold as float
cutoff_threshold = 4.7

# Set the cutoff
highly_rated_products_df_cutoff = cutoff_threshold

# Filter and sort
highly_rated_products_df = top_products_df[
    top_products_df['average_rating'] >= highly_rated_products_df_cutoff
].sort_values('average_rating', ascending=False)

print(f'Cutoff threshold: {cutoff_threshold}')
print(f'Products above cutoff: {len(highly_rated_products_df)}')
print(f'\nTop highly rated products (sorted):')
print(highly_rated_products_df[['title','price','average_rating']].head().to_string())

# Final scatterplot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=highly_rated_products_df, x='price', y='average_rating',
                color='purple', alpha=0.8, s=100)
plt.title(f'Highly Rated Products (Rating ≥ {cutoff_threshold})', fontsize=14)
plt.xlabel('Price ($)', fontsize=12)
plt.ylabel('Average Rating', fontsize=12)
plt.tight_layout()
plt.show()